# Form 144 filings in the weeks before earnings

Match planned insider sales (Form 144) to each company's next earnings 8-K, to see who files to sell ahead of results.

Read it with charts at https://www.quantisnow.com/sec-api/guides/form-144-before-earnings. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

Most companies close the trading window before earnings, so a Form 144 filed in the last weeks of a quarter is either under a 10b5-1 plan or unusual. Pull this month's 144s, then each company's earnings 8-K (Item 2.02), and compare dates.

## This month's large Form 144 filings

### Which insiders filed to sell more than $10 million of stock this month?

A Form 144 comes before the sale, so it shows selling a few days before the Form 4 does.

```shell
curl -X POST https://www.quantisnow.com/api/v1/insider-trades \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "formType:144 AND plannedSale.valueUsd:>10000000 AND filedAt:[2026-11-01 TO 2026-11-24]",
  "from": 0,
  "size": 5,
  "sort": [{"plannedSale.valueUsd": {"order": "desc"}}]
}'
```

Response (real, first 2 of 41):

```json
{
  "total": {
    "value": 41,
    "relation": "eq"
  },
  "data": [
    {
      "id": "0001974078-26-000416:0001206587",
      "accessionNo": "0001974078-26-000416",
      "formType": "144",
      "amendment": false,
      "restatesEarlier": false,
      "filedAt": "2026-11-13T16:28:51-05:00",
      "transactionDate": "2026-11-13",
      "issuer": {
        "cik": "0001660280",
        "name": "Tenable Holdings, Inc.",
        "ticker": "TENB"
      },
      "reportingOwner": {
        "cik": "0001206587",
        "name": "COVIELLO ARTHUR W JR",
        "officerTitle": null,
        "isDirector": true,
        "isOfficer": false,
        "isTenPercentOwner": false
      },
      "transactionCodes": [],
      "purchased": {
        "shares": 0,
        "valueUsd": 0
      },
      "sold": {
        "shares": 0,
        "valueUsd": 0
      },
      "sharesOwnedAfter": null,
      "rule10b51": true,
      "plannedSale": {
        "shares": 12000,
        "valueUsd": 4100000000
      },
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1660280/000197407826000416/0001974078-26-000416-index.htm"
    },
    {
      "id": "0001969223-26-001210:0001622058",
      "accessionNo": "0001969223-26-001210",
      "formType": "144",
      "amendment": false,
      "restatesEarlier": false,
      "filedAt": "2026-11-23T16:53:25-05:00",
      "transactionDate": "2026-11-23",
      "issuer": {
        "cik": "0001668397",
        "name": "Medpace Holdings, Inc.",
        "ticker": "MEDP"
      },
      "reportingOwner": {
        "cik": "0001622058",
        "name": "Troendle August J.",
        "officerTitle": null,
        "isDirector": false,
        "isOfficer": false,
        "isTenPercentOwner": false
      },
      "transactionCodes": [],
      "purchased": {
        "shares": 0,
        "valueUsd": 0
      },
      "sold": {
        "shares": 0,
        "valueUsd": 0
      },
      "sharesOwnedAfter": null,
      "rule10b51": false,
      "plannedSale": {
        "shares": 338225,
        "valueUsd": 236757500
      },
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1668397/000196922326001210/0001969223-26-001210-index.htm"
    }
  ]
}
```

## In Python

In [ ]:
import os, requests

H = {"Authorization": f"Bearer {os.environ['QUANTISNOW_API_KEY']}"}
def search(path, q, size=50, sort=None):
    body = {"query": q, "size": size, **({"sort": sort} if sort else {})}
    return requests.post(f"https://www.quantisnow.com/api/v1/{path}", headers=H, json=body).json()["data"]

planned = search("insider-trades", "formType:144 AND plannedSale.valueUsd:>1000000 AND filedAt:[2026-11-01 TO 2026-11-23]")
for p in planned:
    t = p["issuer"]["ticker"]
    if not t:
        continue
    er = search("8-k/events", f'ticker:{t} AND items.item:"2.02" AND filedAt:>={p["filedAt"][:10]}', size=1, sort="filedAt:asc")
    nxt = er[0]["filedAt"][:10] if er else "none yet"
    print(t, p["reportingOwner"]["name"], p["filedAt"][:10], "plan" if p["rule10b51"] else "no plan", "next earnings:", nxt)

## From the command line

```shell
npx quantisnow insiders --form144 --min 1m --days 30
npx quantisnow events earnings NVDA --days 120
```

Related: [Form 144 API](https://www.quantisnow.com/sec-api/guides/form-144-insider-selling), [10b5-1 plan sales](https://www.quantisnow.com/sec-api/guides/10b5-1-plan-insider-sales), [Earnings 8-K parsing](https://www.quantisnow.com/sec-api/guides/earnings-8-k-2-02).